# Peetre-style decomposition with `psiop` (updated API)

This notebook revisits the Peetre decomposition experiments against the **current** `psiop` package.

Goals:

1. Show how a symbol is split into **local**, **separable**, and **joint** parts in 1D and 2D.
2. Verify that `local + separable + joint` reconstructs the original symbol.
3. Show how `apply()` now **automatically** exploits the decomposition (the old `apply_peetre` is gone).
4. Demonstrate reuse of a cached decomposition across several inputs and boundary conditions.
5. Study the effect of `decompose_order` (asymptotic refinement of joint terms).
6. Run the regression test suite (appendix).

## Peetre-style splitting

For a symbol `p(x, xi)` the package splits, term by term:

- **local** -- polynomial in the frequency variables: `sum_alpha c_alpha(x) xi^alpha`. Applied exactly as variable-coefficient spectral derivatives.
- **separable** -- terms `a(x) q(xi)` with `q` non-polynomial. Applied exactly as weighted Fourier multipliers.
- **joint** -- genuinely entangled space/frequency residual.

### What changed vs the old notebook API

| Old API | Current package |
|---|---|
| `op.peetre_decomposition(taylor_order=..., taylor_x0=..., asymptotic_order=..., refinement_sequence=...)` | `op._get_decomposition(order=...)`, driven by the constructor argument `decompose_order` |
| `op.print_peetre_decomposition(...)` | `op.describe_decomposition(order=..., verbose=True)` |
| `op.apply_peetre(..., apply_joint=True)` | `op.apply(...)` -- always exact; the fast path is engaged automatically when the joint residual is empty |
| `op.apply_peetre(..., apply_joint=False)` | no public equivalent; diagnostic approximations can be assembled from the internal `_apply_local` / `_apply_separable` |
| deco keys `local`, `separable`, `joint_residual`, `local_symbol`, `separable_symbol`, `joint_symbol` | `local_coeffs_sym` (dict monomial -> coefficient), `separable_sym` (list of `(a, q)`), `joint_expr` (sympy expression), plus lambdified `local_funcs`, `separable_funcs`, `joint_symbol_func` |

Taylor refinements are gone; joint terms are refined **only** through the high-frequency asymptotic expansion (`asymptotic_expansion`), and whatever cannot be expanded is kept verbatim in the joint residual.

**Dispatch rule in `apply()` (periodic BC):** joint residual empty -> exact local + separable spectral fast path; otherwise -> full Kohn-Nirenberg backend on the whole symbol (calling the backend on the joint part alone would cost the same). For Dirichlet/Neumann, the whole symbol always goes through the non-periodic backend.

In [ ]:
%reload_ext autoreload
%autoreload 2

import warnings
import time
import numpy as np
import sympy as sp
import matplotlib.pyplot as plt

from psiop_claude import (
    PseudoDifferentialOperator,
    kohn_nirenberg_fft,
    kohn_nirenberg_nonperiodic,
)

In [ ]:
def relative_error(a, b):
    # Relative L2 error with a small safeguard against division by zero.
    return np.linalg.norm(a - b) / (np.linalg.norm(b) + 1e-16)


def make_1d_periodic_grid(N=256, L=2*np.pi):
    # Periodic physical grid and matching FFT frequencies.
    x = np.linspace(-L/2, L/2, N, endpoint=False)
    dx = x[1] - x[0]
    kx = 2*np.pi*np.fft.fftfreq(N, d=dx)
    return x, kx


def make_2d_periodic_grid(Nx=32, Ny=32, Lx=2*np.pi, Ly=2*np.pi):
    # Periodic 2D physical grids and matching FFT frequencies.
    x = np.linspace(-Lx/2, Lx/2, Nx, endpoint=False)
    y = np.linspace(-Ly/2, Ly/2, Ny, endpoint=False)
    dx = x[1] - x[0]
    dy = y[1] - y[0]
    kx = 2*np.pi*np.fft.fftfreq(Nx, d=dx)
    ky = 2*np.pi*np.fft.fftfreq(Ny, d=dy)
    X, Y = np.meshgrid(x, y, indexing='ij')
    return x, y, kx, ky, X, Y


def reconstructed_symbol(op, decomp):
    # Rebuild local + separable + joint from a _get_decomposition() dict.
    # For Kohn-Nirenberg quantization the effective symbol is op.symbol itself.
    xi_vars = (op.xi,) if op.dim == 1 else (op.xi, op.eta)
    local_terms = []
    for monom, coeff in decomp['local_coeffs_sym'].items():
        monomial = sp.prod(v**k for v, k in zip(xi_vars, monom))
        local_terms.append(coeff * monomial)
    local = sp.Add(*local_terms)
    separable = sp.Add(*[a * q for a, q in decomp['separable_sym']])
    joint = decomp['joint_expr']
    return sp.expand(local + separable + joint)

## 1D symbolic decomposition

We build a 1D symbol containing:

- a local polynomial part: `(1 + x**2) * xi**2 + x * xi`,
- a genuinely joint oscillatory part: `sin(x * xi)` (not asymptotically expandable at infinity),
- a non-polynomial mixed part: `exp(-(x - xi)**2)`.

Terms that cannot be asymptotically expanded are kept unchanged in the joint residual. The decomposition is computed eagerly at construction time (`decompose_order`), so expansion warnings may appear there -- this is expected.

In [ ]:
x, xi = sp.symbols('x xi', real=True)

p = (
    (1 + x**2) * xi**2
    + x * xi
    + sp.sin(x * xi)
    + sp.exp(-(x - xi)**2)
)

op = PseudoDifferentialOperator(
    expr=p,
    vars_x=[x],
    mode='symbol',
    decompose_order=3,
)

# Pretty-printed decomposition; also returned as a summary dict.
info = op.describe_decomposition()
print('summary counts:', info['n_local'], 'local |',
      info['n_separable'], 'separable |', info['n_joint'], 'joint')

# Full cached decomposition dict (numerical callables + symbolic pieces).
deco = op._get_decomposition()
print('decomposition keys:', sorted(deco.keys()))

# Symbolic reconstruction check (Kohn-Nirenberg: effective symbol == op.symbol).
error = sp.simplify(sp.expand(reconstructed_symbol(op, deco) - p))
print('Reconstruction error (should simplify to 0):')
sp.pprint(error)

## Numerical application: automatic dispatch

There is no `apply_peetre` anymore. With `boundary_condition='periodic'`, `apply()` dispatches on the decomposition:

- **joint residual empty** -> exact spectral fast path (`_apply_local` + `_apply_separable`), no call to the expensive backend;
- **joint residual non-empty** -> the full Kohn-Nirenberg backend is called on the whole symbol, so the result is bit-for-bit identical to the pre-decomposition behaviour.

`freq_window` / `clamp` / `space_window` only affect the backend (joint) path: the local and separable pieces are exact and need no stabilization. The reference below is the old behaviour: the backend called directly on the full symbol with `is_spatial=True`.

In [ ]:
p_fast = (1 + x**2) * xi**2 + x * xi + sp.sqrt(xi**2 + 1)

op_fast = PseudoDifferentialOperator(p_fast, [x], mode='symbol', decompose_order=3)
op_fast.describe_decomposition()

deco_fast = op_fast._get_decomposition()
print('fast path engaged (joint empty):', deco_fast['joint_symbol_func'] is None)

x_grid, kx = make_1d_periodic_grid(N=256, L=2*np.pi)
u = np.exp(-x_grid**2)

t0 = time.perf_counter()
v_new = op_fast.apply(u, x_grid, kx, boundary_condition='periodic', freq_window=None)
t_new = time.perf_counter() - t0

# Reference = old behaviour: full KN backend on the whole x-dependent symbol.
p_fast_func = sp.lambdify((x, xi), p_fast, 'numpy')
t0 = time.perf_counter()
v_ref = kohn_nirenberg_fft(
    u_vals=u, symbol_func=p_fast_func, x_grid=x_grid, kx=kx,
    fft_func=op_fast.fft, ifft_func=op_fast.ifft, dim=1,
    is_spatial=True, freq_window=None,
)
t_ref = time.perf_counter() - t0

print(f'relative error vs full backend : {relative_error(v_new, v_ref):.3e}')
print(f'decomposition path : {t_new*1e3:.3f} ms')
print(f'full KN backend    : {t_ref*1e3:.3f} ms')
print(f'speed-up           : x{t_ref/max(t_new, 1e-12):.1f}')

In [ ]:
# A symbol with a genuinely entangled part: fallback to the full backend.
p_joint = (1 + x**2) * xi**2 + sp.exp(-((x - xi)**2) / 20)

op_joint = PseudoDifferentialOperator(p_joint, [x], mode='symbol', decompose_order=3)
deco_joint = op_joint._get_decomposition()
print('joint_symbol_func is None ?', deco_joint['joint_symbol_func'] is None)
print('joint residual             :', deco_joint['joint_expr'])

v_new_j = op_joint.apply(u, x_grid, kx, boundary_condition='periodic', freq_window=None)

p_joint_func = sp.lambdify((x, xi), p_joint, 'numpy')
v_ref_j = kohn_nirenberg_fft(
    u_vals=u, symbol_func=p_joint_func, x_grid=x_grid, kx=kx,
    fft_func=op_joint.fft, ifft_func=op_joint.ifft, dim=1,
    is_spatial=True, freq_window=None,
)
print('relative error (fallback vs full backend):', relative_error(v_new_j, v_ref_j))
print('(error is exactly 0: with a non-empty joint residual, apply() calls the')
print(' same full backend on the whole symbol, so the result is unchanged)')

## Purely separable 1D symbol

A symbol of the form `a(x) q(xi)` is detected as separable. `apply()` then applies the Fourier multiplier `q(D)` once and multiplies by `a(x)` -- much cheaper than the space-dependent backend.

In [ ]:
p_sep = sp.exp(-x**2) * sp.sqrt(xi**2 + 1)

op_sep = PseudoDifferentialOperator(p_sep, [x], mode='symbol', decompose_order=3)
op_sep.describe_decomposition()

x_grid_s, kx_s = make_1d_periodic_grid(N=512, L=4*np.pi)
u_sep = np.exp(-x_grid_s**2) * np.cos(5*x_grid_s)

t0 = time.perf_counter()
v_sep = op_sep.apply(u_sep, x_grid_s, kx_s, boundary_condition='periodic', freq_window=None)
t_sep = time.perf_counter() - t0

p_sep_func = sp.lambdify((x, xi), p_sep, 'numpy')
t0 = time.perf_counter()
v_sep_ref = kohn_nirenberg_fft(
    u_vals=u_sep, symbol_func=p_sep_func, x_grid=x_grid_s, kx=kx_s,
    fft_func=op_sep.fft, ifft_func=op_sep.ifft, dim=1,
    is_spatial=True, freq_window=None,
)
t_sep_ref = time.perf_counter() - t0

print(f'relative error       : {relative_error(v_sep, v_sep_ref):.3e}')
print(f'decomposition path   : {t_sep*1e3:.3f} ms | full backend : {t_sep_ref*1e3:.3f} ms')

## Reusing the cached decomposition

The decomposition is computed once (eagerly, at construction time) and cached per `(quantization, weyl_order, order)` in `op._decomposition_cache`. Repeated `apply()` calls -- e.g. in time stepping or parameter studies -- never redo the symbolic work.

In [ ]:
u1 = np.exp(-x_grid**2)
u2 = np.exp(-10*x_grid**2) * np.cos(8*x_grid)
u3 = 1 / np.cosh(x_grid)

t0 = time.perf_counter()
w1 = op_fast.apply(u1, x_grid, kx, boundary_condition='periodic')
w2 = op_fast.apply(u2, x_grid, kx, boundary_condition='periodic')
w3 = op_fast.apply(u3, x_grid, kx, boundary_condition='periodic')
dt = time.perf_counter() - t0

print('cached decomposition keys:', list(op_fast._decomposition_cache.keys()))
print(f'3 successive apply() calls: {dt*1e3:.3f} ms total (decomposition computed only once)')
print('norms:', np.linalg.norm(w1), np.linalg.norm(w2), np.linalg.norm(w3))

## Effect of `decompose_order` (replaces the Taylor-order study)

The old notebook studied the accuracy of a spatial Taylor approximation of the joint part. In the current package, joint terms are refined only via the **high-frequency asymptotic expansion**, and `decompose_order` controls how many terms are peeled off:

- each peeled-off term is reclassified (typically into the separable bucket),
- the unexpanded remainder is kept **exactly** in the joint residual.

Since `apply()` is always exact, the plot below measures, as a diagnostic, the error of the *local + separable only* approximation (joint remainder dropped), assembled from the internal building blocks `_apply_local` / `_apply_separable`.

In [ ]:
p_conv = (1 + x**2) * xi**2 + xi / (xi**2 + x**2 + 1)

op_conv = PseudoDifferentialOperator(p_conv, [x], mode='symbol', decompose_order=1)

# Exact reference (joint non-empty -> full backend).
v_ref_conv = op_conv.apply(u, x_grid, kx, boundary_condition='periodic', freq_window=None)

errors_conv, n_joints = [], []
orders = [0, 1, 2, 3, 4, 5, 6]

for order in orders:
    deco_k = op_conv._get_decomposition(order=order)
    # Diagnostic approximation: keep only the exact local + separable pieces,
    # drop the asymptotic remainder left in the joint residual. This uses the
    # internal building blocks of apply() (there is no public apply_joint=False
    # anymore, since apply() is always exact).
    v_approx = (
        op_conv._apply_local(u, x_grid, kx, None, None, None, deco_k['local_funcs'])
        + op_conv._apply_separable(u, x_grid, kx, None, None, None, deco_k['separable_funcs'])
    )
    err = relative_error(v_approx, v_ref_conv)
    je = deco_k['joint_expr']
    n_joint = 0 if je == 0 else len(sp.Add.make_args(je))
    errors_conv.append(err)
    n_joints.append(n_joint)
    print(f'order {order}: joint terms = {n_joint}, approx relative error = {err:.3e}')

plt.figure(figsize=(7, 4))
plt.semilogy(orders, errors_conv, 'o-')
plt.xlabel('decompose_order (asymptotic refinement of joint terms)')
plt.ylabel('Relative error of local+separable approximation')
plt.title('Approximation error vs decompose_order')
plt.grid(True, which='both', ls='--', alpha=0.5)
plt.show()

## Non-periodic (Dirichlet) application

The local/separable fast paths rely on periodic spectral differentiation, so they are only used for `boundary_condition='periodic'`. For Dirichlet/Neumann, `apply()` passes the whole effective symbol to the non-periodic Kohn-Nirenberg backend, unchanged.

In [ ]:
N_dir = 256
x_dir = np.linspace(-5.0, 5.0, N_dir, endpoint=False)
xi_dir = np.linspace(-30.0, 30.0, N_dir)
u_dir = np.exp(-x_dir**2)

p_dir = (1 + x**2) * xi**2 + x * xi + sp.sin(x * xi)
op_dir = PseudoDifferentialOperator(p_dir, [x], mode='symbol', decompose_order=3)

v_dir = op_dir.apply(u_dir, x_dir, xi_dir, boundary_condition='dirichlet',
                     freq_window='gaussian', clamp=1e6)

p_dir_func = sp.lambdify((x, xi), p_dir, 'numpy')
v_dir_ref = kohn_nirenberg_nonperiodic(
    u_vals=u_dir, x_grid=x_dir, xi_grid=xi_dir,
    symbol_func=p_dir_func, is_spatial=True,
    freq_window='gaussian', clamp=1e6,
)
print('Dirichlet relative error vs reference backend:', relative_error(v_dir, v_dir_ref))

## Weyl quantization

For `quantization='weyl'`, the same decomposition machinery is applied to the **KN-equivalent** symbol obtained from `weyl_to_kn_symbol(order=weyl_order)`. For symbols polynomial in `xi` the conversion series is exact and finite. Note: for Weyl operators the decomposition reconstructs the KN-equivalent symbol, not `op.symbol`.

In [ ]:
p_weyl = x * xi + xi**2
op_weyl = PseudoDifferentialOperator(p_weyl, [x], mode='symbol',
                                     quantization='weyl', decompose_order=3)
op_weyl.describe_decomposition(weyl_order=4)

v_weyl = op_weyl.apply(u, x_grid, kx, boundary_condition='periodic',
                       freq_window=None, weyl_order=4)

kn_eq = op_weyl.weyl_to_kn_symbol(order=4)
print('KN-equivalent symbol:', kn_eq)
kn_func = sp.lambdify((x, xi), kn_eq, 'numpy')
v_weyl_ref = kohn_nirenberg_fft(
    u_vals=u, symbol_func=kn_func, x_grid=x_grid, kx=kx,
    fft_func=op_weyl.fft, ifft_func=op_weyl.ifft, dim=1,
    is_spatial=True, freq_window=None,
)
print('Weyl relative error vs KN reference:', relative_error(v_weyl, v_weyl_ref))

## 2D symbolic decomposition

The same idea extends to two spatial variables: local polynomial terms in `(xi, eta)`, separable terms `a(x, y) q(xi, eta)`, and a joint residual. The oscillatory term `sin(x*xi + y*eta)` is not asymptotically expandable at infinity and stays in the joint residual.

In [ ]:
x, y, xi, eta = sp.symbols('x y xi eta', real=True)

p2 = (1 + x**2) * xi**2 + y**2 * eta**2 + sp.sin(x*xi + y*eta)

op2 = PseudoDifferentialOperator(p2, [x, y], mode='symbol', decompose_order=3)
op2.describe_decomposition()

deco2 = op2._get_decomposition()
error2 = sp.simplify(sp.expand(reconstructed_symbol(op2, deco2) - p2))
print('2D reconstruction error (should simplify to 0):')
sp.pprint(error2)

## 2D numerical application

For 2D operators the full space-dependent backend is a high-dimensional quadrature (O(N^4)), so the decomposition fast path is especially valuable. Grids are kept modest to keep runtimes reasonable.

In [ ]:
p2_fast = (1 + x**2) * xi**2 + y**2 * eta**2 + sp.sqrt(xi**2 + eta**2 + 1)

op2_fast = PseudoDifferentialOperator(p2_fast, [x, y], mode='symbol', decompose_order=3)
deco2_fast = op2_fast._get_decomposition()
print('fast path engaged (joint empty):', deco2_fast['joint_symbol_func'] is None)

x2, y2, kx2, ky2, X2, Y2 = make_2d_periodic_grid(Nx=32, Ny=32, Lx=16.0, Ly=16.0)
u2 = np.exp(-(X2**2 + Y2**2) / 4)

t0 = time.perf_counter()
v2_new = op2_fast.apply(u2, x2, kx2, boundary_condition='periodic',
                        y_grid=y2, ky=ky2, freq_window=None)
t2_new = time.perf_counter() - t0

p2_fast_func = sp.lambdify((x, y, xi, eta), p2_fast, 'numpy')
t0 = time.perf_counter()
v2_ref = kohn_nirenberg_fft(
    u_vals=u2, symbol_func=p2_fast_func, x_grid=x2, kx=kx2,
    fft_func=op2_fast.fft, ifft_func=op2_fast.ifft, dim=2,
    y_grid=y2, ky=ky2, is_spatial=True, freq_window=None,
)
t2_ref = time.perf_counter() - t0

print(f'relative error       : {relative_error(v2_new, v2_ref):.3e}')
print(f'decomposition path   : {t2_new*1e3:.2f} ms | full backend : {t2_ref*1e3:.2f} ms')
print(f'speed-up             : x{t2_ref/max(t2_new, 1e-12):.1f}')

In [ ]:
# 2D symbol with a joint residual: exact apply vs diagnostic local+separable
# approximation (internal building blocks), to visualize what the joint part does.
p2_vis = (1 + x**2) * xi**2 + y**2 * eta**2 + sp.exp(-((x - xi/4)**2 + (y - eta/4)**2) / 30)

op2_vis = PseudoDifferentialOperator(p2_vis, [x, y], mode='symbol', decompose_order=3)
op2_vis.describe_decomposition()
deco2_vis = op2_vis._get_decomposition()

t0 = time.perf_counter()
v2_full = op2_vis.apply(u2, x2, kx2, boundary_condition='periodic',
                        y_grid=y2, ky=ky2, freq_window=None)
t2_full = time.perf_counter() - t0

t0 = time.perf_counter()
v2_approx = (
    op2_vis._apply_local(u2, x2, kx2, y2, ky2, None, deco2_vis['local_funcs'])
    + op2_vis._apply_separable(u2, x2, kx2, y2, ky2, None, deco2_vis['separable_funcs'])
)
t2_approx = time.perf_counter() - t0

print(f'full apply: {t2_full*1e3:.1f} ms | local+separable approx: {t2_approx*1e3:.1f} ms')
print('global relative error:', relative_error(v2_approx, v2_full))

vmax = float(np.max(np.abs(v2_full.real)))
vmax = max(vmax, 1e-12)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

a = axes[0].pcolormesh(X2, Y2, v2_full.real, shading='auto',
                       cmap='viridis', vmin=-vmax, vmax=vmax)
axes[0].set_title('Full apply (exact)')
fig.colorbar(a, ax=axes[0], fraction=0.046)

b = axes[1].pcolormesh(X2, Y2, v2_approx.real, shading='auto',
                       cmap='viridis', vmin=-vmax, vmax=vmax)
axes[1].set_title('Local + separable approximation')
fig.colorbar(b, ax=axes[1], fraction=0.046)

c = axes[2].pcolormesh(X2, Y2, np.abs(v2_full.real - v2_approx.real),
                       shading='auto', cmap='inferno')
axes[2].set_title('|error| (dropped joint residual)')
fig.colorbar(c, ax=axes[2], fraction=0.046)

for ax in axes:
    ax.set_xlabel('x')
    ax.set_ylabel('y')
    ax.set_aspect('equal')

plt.suptitle('2D: exact apply vs local+separable approximation')
plt.tight_layout()
plt.show()

## Timing benchmark (1D, N=1024)

The main practical advantage of the decomposition is speed when the joint residual is empty: the local and separable parts are applied with a handful of FFTs instead of the O(N^2) space-dependent quadrature.

In [ ]:
x_grid_t, kx_t = make_1d_periodic_grid(N=1024, L=2*np.pi)
u_t = np.exp(-x_grid_t**2)

p_time = (1 + x**2) * xi**2 + sp.sqrt(xi**2 + 1)
op_time = PseudoDifferentialOperator(p_time, [x], mode='symbol', decompose_order=3)
deco_time = op_time._get_decomposition()
print('fast path engaged:', deco_time['joint_symbol_func'] is None)

t0 = time.perf_counter()
v_time_new = op_time.apply(u_t, x_grid_t, kx_t, boundary_condition='periodic', freq_window=None)
t_new_t = time.perf_counter() - t0

p_time_func = sp.lambdify((x, xi), p_time, 'numpy')
t0 = time.perf_counter()
v_time_ref = kohn_nirenberg_fft(
    u_vals=u_t, symbol_func=p_time_func, x_grid=x_grid_t, kx=kx_t,
    fft_func=op_time.fft, ifft_func=op_time.ifft, dim=1,
    is_spatial=True, freq_window=None,
)
t_ref_t = time.perf_counter() - t0

print(f'decomposition path : {t_new_t*1e3:.3f} ms')
print(f'full KN backend    : {t_ref_t*1e3:.3f} ms')
print(f'speed-up           : x{t_ref_t/max(t_new_t, 1e-12):.1f}')
print(f'relative error     : {relative_error(v_time_new, v_time_ref):.3e}')

## Practical guide

Important options and facts:

- `decompose_order` (constructor argument, default 3): asymptotic order used to peel local/separable pieces off joint terms.
- `op.describe_decomposition(order=..., weyl_order=...)`: pretty-print and return a summary dict (`local_coeffs`, `separable`, `joint_residual`, counts).
- `op._get_decomposition(order=...)`: the full cached dict, including lambdified callables and symbolic pieces.
- `op.apply()` is **always exact**; the decomposition only decides *how fast* the result is obtained:
  - periodic + joint empty -> exact spectral fast path,
  - periodic + joint non-empty -> full backend on the whole symbol,
  - Dirichlet/Neumann -> full non-periodic backend, decomposition not used numerically.
- `freq_window` / `clamp` / `space_window` only affect the backend path; local/separable pieces are exact and unwindowed.
- `dealiasing_mask` is applied consistently in every Fourier multiplication.
- For Weyl quantization, the decomposition is applied to the KN-converted symbol (`weyl_order` controls the conversion; exact and finite for polynomial symbols).

Recommended workflow:

1. Build the operator with a suitable `decompose_order`.
2. Inspect with `describe_decomposition()`; check `joint_symbol_func is None` to know whether the fast path will be engaged.
3. Call `apply()` normally -- dispatch is automatic and cached.
4. If a cheap *approximation* is acceptable (exploratory runs), assemble `_apply_local + _apply_separable` manually, knowing the joint remainder is dropped (diagnostic use only).

## Summary

The Peetre decomposition still provides a structured way to split a pseudo-differential symbol into computationally useful pieces:

- the **local** part behaves like a variable-coefficient differential operator,
- the **separable** part behaves like a spatial amplitude times a Fourier multiplier,
- the **joint** part is the genuinely nonlocal, space-frequency coupled remainder.

Compared to the old experiments, the API is simpler and stricter: no Taylor machinery, no user-facing approximate application. `apply()` is always exact; the decomposition is a pure *acceleration* (exact spectral shortcuts whenever the joint residual is empty), and joint terms are only refined through high-frequency asymptotic expansions whose remainders are kept exactly.

## Appendix

1. Detailed decomposition inspection for four representative 2D symbols (local/non-local x constant/variable coefficients).
2. Regression test suite (T1-T14) comparing `apply()` against the pre-decomposition backend behaviour.
3. Low-level debug cell for `_classify_terms`.

In [ ]:
from sympy import Rational, pprint, sin, cos

x, y = sp.symbols('x y', real=True)
xi, eta = sp.symbols('xi eta', real=True)

# 1. Local, constant coefficients -> FFT fast path expected.
sym_2d_loc_c = xi**2 + eta**2
# 2. Local, variable coefficients -> space-dependent path expected.
sym_2d_loc_v = (1 + Rational(1, 2) * sin(x) * cos(y)) * (xi**2 + eta**2)
# 3. Non-local, constant coefficients (fractional-Laplacian-like).
sym_2d_nloc_c = (xi**2 + eta**2)**Rational(3, 4)
# 4. Non-local, variable coefficients.
sym_2d_nloc_v = (1 + Rational(1, 2) * sin(x) * cos(y)) * (xi**2 + eta**2)**Rational(3, 4)

ops_2d = {
    '2D Local (Const)': PseudoDifferentialOperator(sym_2d_loc_c, [x, y], mode='symbol', decompose_order=4),
    '2D Local (Var)': PseudoDifferentialOperator(sym_2d_loc_v, [x, y], mode='symbol', decompose_order=4),
    '2D Non-Local (Const)': PseudoDifferentialOperator(sym_2d_nloc_c, [x, y], mode='symbol', decompose_order=4),
    '2D Non-Local (Var)': PseudoDifferentialOperator(sym_2d_nloc_v, [x, y], mode='symbol', decompose_order=4),
}


def has_var(expr, name):
    # Robust check by symbol name (the constructor substitutes internal xi/eta).
    return any(s.name == name for s in expr.free_symbols)


for name, op in ops_2d.items():
    print('=' * 100)
    print(f"Operator: {name}")
    print('=' * 100)
    print('Symbol:')
    pprint(op.symbol)

    print('Basic information:')
    print(f"  dim                 : {op.dim}")
    print(f"  spatially dependent : {has_var(op.symbol, 'x') or has_var(op.symbol, 'y')}")
    print(f"  depends on xi       : {has_var(op.symbol, 'xi')}")
    print(f"  depends on eta      : {has_var(op.symbol, 'eta')}")

    print()
    print('-' * 100)
    print('describe_decomposition(order=4)')
    print('-' * 100)
    info = op.describe_decomposition(order=4)
    deco = op._get_decomposition(order=4)

    print()
    print('-' * 100)
    print('Detailed inspection')
    print('-' * 100)
    print(f"Number of local terms     : {info['n_local']}")
    print(f"Number of separable terms : {info['n_separable']}")
    print(f"Number of joint terms     : {info['n_joint']}")

    print('Local polynomial coefficients:')
    if info['local_coeffs']:
        xi_vars = (op.xi, op.eta)
        for monom, coeff in info['local_coeffs'].items():
            monomial = sp.prod(v**k for v, k in zip(xi_vars, monom))
            print(f"  monomial {monom}: coeff = {coeff}, term = {sp.expand(coeff * monomial)}")
    else:
        print('  none')

    print('Separable terms a(x, y) * q(xi, eta):')
    if info['separable']:
        for i, (a, q) in enumerate(info['separable'], start=1):
            print(f"  term {i}: a = {a}")
            print(f"           q = {q}")
    else:
        print('  none')

    print('Joint residual terms:')
    if info['joint_residual']:
        for i, t in enumerate(info['joint_residual'], start=1):
            print(f"  term {i}: {t}")
    else:
        print('  none')

    reconstructed = reconstructed_symbol(op, deco)
    reconstruction_error = sp.simplify(sp.expand(reconstructed - op.symbol))
    print('Reconstruction check (local + separable + joint - symbol):')
    pprint(reconstruction_error)
    if reconstruction_error == 0:
        print('Reconstruction is exact.')
    else:
        print('Simplification did not reduce the error to zero (may still be equivalent).')
    print()

In [ ]:
# ----------------------------------------------------------------------
# Regression test suite for the local / separable / joint decomposition.
# 14 cases (T1-T8 in 1D, T9-T14 in 2D), each isolating one structural
# branch, crossed with boundary condition and quantization. Each case
# compares apply() against an independent reference that calls the KN
# backend directly on the full (effective) symbol, i.e. the
# pre-decomposition behaviour: the decomposition must never change the
# numerical result, only how fast it is obtained.
# ----------------------------------------------------------------------
import warnings
import time
import numpy as np
import sympy as sp

from psiop import (
    PseudoDifferentialOperator,
    kohn_nirenberg_fft,
    kohn_nirenberg_nonperiodic,
)

warnings.filterwarnings('ignore')

TOL = 1e-8
_results = []


def _check(name, passed, detail=''):
    _results.append((name, passed, detail))
    status = 'PASS' if passed else 'FAIL'
    print(f'[{status}] {name}' + (f'  -- {detail}' if detail else ''))


def rel_err(a, b):
    nb = np.linalg.norm(b)
    return np.linalg.norm(a - b) / nb if nb > 0 else np.linalg.norm(a - b)


# ---------------------------- 1D setup ---------------------------------
x, xi = sp.symbols('x xi', real=True)

N1 = 128
L1 = 10.0
xg1 = np.linspace(-L1, L1, N1, endpoint=False)
kg1 = 2 * np.pi * np.fft.fftfreq(N1, d=(xg1[1] - xg1[0]))
u1 = np.exp(-xg1**2 / 4)


def ref_periodic_1d(op, p_expr, **kw):
    f = sp.lambdify((x, xi), p_expr, 'numpy')
    return kohn_nirenberg_fft(u_vals=u1, symbol_func=f, x_grid=xg1, kx=kg1,
                              fft_func=op.fft, ifft_func=op.ifft, dim=1,
                              is_spatial=True, **kw)


def ref_dirichlet_1d(op, p_expr, **kw):
    f = sp.lambdify((x, xi), p_expr, 'numpy')
    return kohn_nirenberg_nonperiodic(u_vals=u1, x_grid=xg1, xi_grid=kg1,
                                      symbol_func=f, is_spatial=True, **kw)


# T1: local only, constant coefficients
p1 = xi**2
op1 = PseudoDifferentialOperator(p1, [x], decompose_order=3)
decomp1 = op1._get_decomposition(order=3)
t0 = time.time()
r1 = op1.apply(u1, xg1, kg1, boundary_condition='periodic', freq_window=None)
t_new1 = time.time() - t0
t0 = time.time()
r1_ref = ref_periodic_1d(op1, p1, freq_window=None)
t_ref1 = time.time() - t0
_check('T1 local-only-1D precision', rel_err(r1, r1_ref) < TOL, f'err={rel_err(r1, r1_ref):.2e}')
_check('T1 local-only-1D joint empty', decomp1['joint_expr'] == 0)
_check('T1 local-only-1D timing', t_new1 <= t_ref1 * 1.5 + 1e-3,
       f'new={t_new1*1000:.3f}ms ref={t_ref1*1000:.3f}ms')

# T2: local only, x-dependent coefficients
p2 = (1 + x**2) * xi**2 + x * xi
op2t = PseudoDifferentialOperator(p2, [x], decompose_order=3)
decomp2 = op2t._get_decomposition(order=3)
r2 = op2t.apply(u1, xg1, kg1, boundary_condition='periodic', freq_window=None)
r2_ref = ref_periodic_1d(op2t, p2, freq_window=None)
_check('T2 local-xdep-1D precision', rel_err(r2, r2_ref) < TOL, f'err={rel_err(r2, r2_ref):.2e}')
_check('T2 local-xdep-1D joint empty', decomp2['joint_expr'] == 0)

# T3: separable only
p3 = sp.sqrt(sp.Abs(xi)) + 1 / (1 + xi**2)
op3 = PseudoDifferentialOperator(p3, [x], decompose_order=3)
decomp3 = op3._get_decomposition(order=3)
t0 = time.time()
r3 = op3.apply(u1, xg1, kg1, boundary_condition='periodic', freq_window=None)
t_new3 = time.time() - t0
t0 = time.time()
r3_ref = ref_periodic_1d(op3, p3, freq_window=None)
t_ref3 = time.time() - t0
_check('T3 separable-only-1D precision', rel_err(r3, r3_ref) < TOL, f'err={rel_err(r3, r3_ref):.2e}')
_check('T3 separable-only-1D joint empty', decomp3['joint_expr'] == 0)
_check('T3 separable-only-1D timing', t_new3 <= t_ref3 * 1.5 + 1e-3,
       f'new={t_new3*1000:.3f}ms ref={t_ref3*1000:.3f}ms')

# T4: local + separable, joint EMPTY + dealiasing mask + freq_window invariance
p4 = (1 + x**2) * xi**2 + sp.sqrt(sp.Abs(xi))
op4 = PseudoDifferentialOperator(p4, [x], decompose_order=3)
decomp4 = op4._get_decomposition(order=3)
t0 = time.time()
r4 = op4.apply(u1, xg1, kg1, boundary_condition='periodic', freq_window=None)
t_new4 = time.time() - t0
t0 = time.time()
r4_ref = ref_periodic_1d(op4, p4, freq_window=None)
t_ref4 = time.time() - t0
_check('T4 local+separable-1D precision', rel_err(r4, r4_ref) < TOL, f'err={rel_err(r4, r4_ref):.2e}')
_check('T4 local+separable-1D joint empty', decomp4['joint_expr'] == 0)
_check('T4 local+separable-1D timing (speed-up)', t_new4 < t_ref4,
       f'new={t_new4*1000:.3f}ms ref={t_ref4*1000:.3f}ms')

mask = (np.abs(kg1) < 0.6 * np.max(np.abs(kg1))).astype(float)
r4_mask = op4.apply(u1, xg1, kg1, boundary_condition='periodic', freq_window=None, dealiasing_mask=mask)
r4_mask_ref = ref_periodic_1d(op4, p4, freq_window=None, dealiasing_mask=mask)
_check('T4 dealiasing_mask consistency', rel_err(r4_mask, r4_mask_ref) < TOL,
       f'err={rel_err(r4_mask, r4_mask_ref):.2e}')

r4_win = op4.apply(u1, xg1, kg1, boundary_condition='periodic', freq_window='gaussian')
_check('T4 freq_window invariance (local/separable exact, no windowing applied)',
       rel_err(r4_win, r4) < TOL, f'err={rel_err(r4_win, r4):.2e}')

# T5: joint NON-EMPTY -> fallback must reproduce the old behaviour exactly
p5 = xi**2 + sp.exp(-((x - xi)**2) / 20)
op5 = PseudoDifferentialOperator(p5, [x], decompose_order=4)
decomp5 = op5._get_decomposition(order=4)
r5 = op5.apply(u1, xg1, kg1, boundary_condition='periodic', freq_window=None)
r5_ref = ref_periodic_1d(op5, p5, freq_window=None)
_check('T5 joint-nonempty-1D precision (fallback)', rel_err(r5, r5_ref) < TOL,
       f'err={rel_err(r5, r5_ref):.2e}')
_check('T5 joint-nonempty-1D joint indeed nonempty', decomp5['joint_expr'] != 0)

# T6: Weyl quantization (polynomial symbol -> exact finite correction)
p6 = x * xi + xi**2
op6 = PseudoDifferentialOperator(p6, [x], quantization='weyl', decompose_order=3)
r6 = op6.apply(u1, xg1, kg1, boundary_condition='periodic', freq_window=None, weyl_order=4)
kn_eq6 = op6.weyl_to_kn_symbol(order=4)
r6_ref = ref_periodic_1d(op6, kn_eq6, freq_window=None)
_check('T6 weyl-1D precision', rel_err(r6, r6_ref) < TOL, f'err={rel_err(r6, r6_ref):.2e}')

# T7: Dirichlet BC, mixed symbol (path untouched by the decomposition)
p7 = xi**2 + sp.sqrt(sp.Abs(xi)) + sp.exp(-((x - xi)**2) / 20)
op7 = PseudoDifferentialOperator(p7, [x], decompose_order=4)
r7 = op7.apply(u1, xg1, kg1, boundary_condition='dirichlet')
r7_ref = ref_dirichlet_1d(op7, p7)
_check('T7 dirichlet-1D precision', rel_err(r7, r7_ref) < TOL, f'err={rel_err(r7, r7_ref):.2e}')

# T8: robustness with a very low decompose_order (no crash, correct result)
op8 = PseudoDifferentialOperator(p5, [x], decompose_order=1)
try:
    r8 = op8.apply(u1, xg1, kg1, boundary_condition='periodic', freq_window=None)
    r8_ref = ref_periodic_1d(op8, p5, freq_window=None)
    _check('T8 robustness (low decompose_order, no crash)', rel_err(r8, r8_ref) < TOL,
           f'err={rel_err(r8, r8_ref):.2e}')
except Exception as e:
    _check('T8 robustness (low decompose_order, no crash)', False, f'raised {e!r}')


# ---------------------------- 2D setup ---------------------------------
y, eta = sp.symbols('y eta', real=True)

N2 = 32
L2 = 8.0
xg2 = np.linspace(-L2, L2, N2, endpoint=False)
yg2 = np.linspace(-L2, L2, N2, endpoint=False)
kgx2 = 2 * np.pi * np.fft.fftfreq(N2, d=(xg2[1] - xg2[0]))
kgy2 = 2 * np.pi * np.fft.fftfreq(N2, d=(yg2[1] - yg2[0]))
XG2, YG2 = np.meshgrid(xg2, yg2, indexing='ij')
u2t = np.exp(-(XG2**2 + YG2**2) / 4)


def ref_periodic_2d(op, p_expr, **kw):
    f = sp.lambdify((x, y, xi, eta), p_expr, 'numpy')
    return kohn_nirenberg_fft(u_vals=u2t, symbol_func=f, x_grid=xg2, kx=kgx2,
                              fft_func=op.fft, ifft_func=op.ifft, dim=2,
                              y_grid=yg2, ky=kgy2, is_spatial=True, **kw)


def ref_dirichlet_2d(op, p_expr, **kw):
    f = sp.lambdify((x, y, xi, eta), p_expr, 'numpy')
    return kohn_nirenberg_nonperiodic(u_vals=u2t, x_grid=(xg2, yg2),
                                      xi_grid=(kgx2, kgy2), symbol_func=f,
                                      is_spatial=True, **kw)


# T9: local only, 2D
p9 = xi**2 + eta**2
op9 = PseudoDifferentialOperator(p9, [x, y], decompose_order=3)
decomp9 = op9._get_decomposition(order=3)
t0 = time.time()
r9 = op9.apply(u2t, xg2, kgx2, boundary_condition='periodic', y_grid=yg2, ky=kgy2, freq_window=None)
t_new9 = time.time() - t0
t0 = time.time()
r9_ref = ref_periodic_2d(op9, p9, freq_window=None)
t_ref9 = time.time() - t0
_check('T9 local-only-2D precision', rel_err(r9, r9_ref) < TOL, f'err={rel_err(r9, r9_ref):.2e}')
_check('T9 local-only-2D joint empty', decomp9['joint_expr'] == 0)
_check('T9 local-only-2D timing (speed-up)', t_new9 < t_ref9,
       f'new={t_new9*1000:.2f}ms ref={t_ref9*1000:.2f}ms')

# T10: separable only, 2D
p10 = sp.sqrt(xi**2 + eta**2 + 1)
op10 = PseudoDifferentialOperator(p10, [x, y], decompose_order=3)
decomp10 = op10._get_decomposition(order=3)
t0 = time.time()
r10 = op10.apply(u2t, xg2, kgx2, boundary_condition='periodic', y_grid=yg2, ky=kgy2, freq_window=None)
t_new10 = time.time() - t0
t0 = time.time()
r10_ref = ref_periodic_2d(op10, p10, freq_window=None)
t_ref10 = time.time() - t0
_check('T10 separable-only-2D precision', rel_err(r10, r10_ref) < TOL, f'err={rel_err(r10, r10_ref):.2e}')
_check('T10 separable-only-2D joint empty', decomp10['joint_expr'] == 0)
_check('T10 separable-only-2D timing (speed-up)', t_new10 < t_ref10,
       f'new={t_new10*1000:.2f}ms ref={t_ref10*1000:.2f}ms')

# T11: local + separable, joint EMPTY, 2D
p11 = xi**2 + eta**2 + sp.sqrt(xi**2 + eta**2 + 1)
op11 = PseudoDifferentialOperator(p11, [x, y], decompose_order=3)
decomp11 = op11._get_decomposition(order=3)
t0 = time.time()
r11 = op11.apply(u2t, xg2, kgx2, boundary_condition='periodic', y_grid=yg2, ky=kgy2, freq_window=None)
t_new11 = time.time() - t0
t0 = time.time()
r11_ref = ref_periodic_2d(op11, p11, freq_window=None)
t_ref11 = time.time() - t0
_check('T11 local+separable-2D precision', rel_err(r11, r11_ref) < TOL, f'err={rel_err(r11, r11_ref):.2e}')
_check('T11 local+separable-2D joint empty', decomp11['joint_expr'] == 0)
_check('T11 local+separable-2D timing (speed-up)', t_new11 < t_ref11,
       f'new={t_new11*1000:.2f}ms ref={t_ref11*1000:.2f}ms')

# T12: joint NON-EMPTY, 2D -> fallback
p12 = xi**2 + eta**2 + sp.exp(-((x - xi/4)**2 + (y - eta/4)**2) / 30)
op12 = PseudoDifferentialOperator(p12, [x, y], decompose_order=3)
decomp12 = op12._get_decomposition(order=3)
r12 = op12.apply(u2t, xg2, kgx2, boundary_condition='periodic', y_grid=yg2, ky=kgy2, freq_window=None)
r12_ref = ref_periodic_2d(op12, p12, freq_window=None)
_check('T12 joint-nonempty-2D precision (fallback)', rel_err(r12, r12_ref) < TOL,
       f'err={rel_err(r12, r12_ref):.2e}')
_check('T12 joint-nonempty-2D joint indeed nonempty', decomp12['joint_expr'] != 0)

# T13: Weyl quantization, 2D
p13 = x * xi + y * eta + xi**2 + eta**2
op13 = PseudoDifferentialOperator(p13, [x, y], quantization='weyl', decompose_order=3)
r13 = op13.apply(u2t, xg2, kgx2, boundary_condition='periodic', y_grid=yg2, ky=kgy2,
                 freq_window=None, weyl_order=4)
kn_eq13 = op13.weyl_to_kn_symbol(order=4)
r13_ref = ref_periodic_2d(op13, kn_eq13, freq_window=None)
_check('T13 weyl-2D precision', rel_err(r13, r13_ref) < TOL, f'err={rel_err(r13, r13_ref):.2e}')

# T14: Dirichlet BC, 2D
p14 = xi**2 + eta**2 + sp.sqrt(xi**2 + eta**2 + 1)
op14 = PseudoDifferentialOperator(p14, [x, y], decompose_order=3)
r14 = op14.apply(u2t, xg2, kgx2, boundary_condition='dirichlet', y_grid=yg2, ky=kgy2)
r14_ref = ref_dirichlet_2d(op14, p14)
_check('T14 dirichlet-2D precision', rel_err(r14, r14_ref) < TOL, f'err={rel_err(r14, r14_ref):.2e}')


# ----------------------------- summary ---------------------------------
n_pass = sum(1 for _, ok, _ in _results if ok)
n_total = len(_results)
print(f'\n{n_pass}/{n_total} checks passed.')
if n_pass != n_total:
    print('FAILED checks:')
    for name, ok, detail in _results:
        if not ok:
            print(f'  - {name}: {detail}')

In [ ]:
# Low-level inspection of the classification step.
x, xi = sp.symbols('x xi', real=True)
p_dbg = xi**2

op_dbg = PseudoDifferentialOperator(p_dbg, [x], decompose_order=3)
decomp_dbg = op_dbg._get_decomposition(order=3)

print('joint_expr =', repr(decomp_dbg['joint_expr']))
print('joint_expr == 0 ?', decomp_dbg['joint_expr'] == 0)
print('local_funcs:', decomp_dbg['local_funcs'])
print('separable_funcs:', decomp_dbg['separable_funcs'])
print('op.xi:', op_dbg.xi, type(op_dbg.xi), op_dbg.xi.assumptions0)
print('symbol:', repr(op_dbg.symbol))

lc, sep, joint = op_dbg._classify_terms(op_dbg.symbol)
print('classify_terms -> local:', lc, '| separable:', sep, '| joint:', joint)